# M16 — Pretraining, frozen features, and actual fine-tuning

**Original guided lab · 75–100 minutes.** Read 20 min, predict/code 35 min, failure investigation 20 min, explain and transfer 15 min. Run all cells in order in a fresh kernel. All executed data are synthetic unless explicitly stated. No network, GPU, or external dataset is required.

Pretraining learns a representation before the downstream target is fitted. Supervised pretraining uses labels for an earlier task. Self-supervised pretraining creates a learning target from the data themselves, such as predicting withheld information or reconstructing a corrupted observation. Unsupervised reconstruction is one such representation-learning route, but it is not the same as every modern contrastive or masked-image method.

The key distinction for this notebook is which parameters change. A frozen encoder produces features without updating its weights; a new head learns the target from those features. Fine-tuning updates some or all encoder weights along with the head. A frozen encoder can still encode nuisance variation, and fine-tuning can overfit a small downstream cohort. Neither strategy removes the need for independent evaluation or overlap checks against the pretraining population.

We build a small linear denoising autoencoder. Three latent variables generate twelve measured features. The encoder compresses corrupted measurements into three coordinates, and the decoder reconstructs the clean measurement target. Both matrices are actually optimized by gradient descent. We use one set of participants for pretraining, a separate set for downstream head fitting, and another for evaluation. The clean target exists because we generated the data; in real self-supervision, the target construction and corruption process require justification.

After pretraining, logistic regression learns a binary target from frozen features. We copy the encoder and verify that head fitting has not changed it. Then a separate fine-tuning experiment updates an encoder copy and a logistic head using only downstream training labels. We check both loss reduction and actual encoder change. A difference between frozen and fine-tuned test scores is descriptive for this fixed demonstration, not a license to choose whichever procedure wins on repeated inspections of the test data.

Reconstruction objectives retain what helps reconstruct observations, which can include scanner patterns and nuisance. The downstream target may depend on information discarded by the bottleneck. Conversely, a richer representation can allow a small head to learn with fewer labels. Compare with simple feature baselines and inspect where transfer succeeds or fails rather than assuming pretraining is beneficial by definition.

Modern contrastive objectives use relationships between augmented views; masked objectives predict omitted content. Augmentations must preserve the scientific target. A left–right flip, contrast change, or spatial crop can be inappropriate for a lateralized or lesion-specific task. Ask AI to justify each augmentation in biological terms and to distinguish the code license, checkpoint terms, source-data restrictions, and intended-use evidence before loading a real imaging encoder.

## Transformation contract

Corrupted measurements → learned encoder bottleneck → learned reconstruction; later frozen embeddings → fitted head. Fine-tuning additionally changes the encoder. The bottleneck discards variation; pretraining provenance and target-preserving augmentations determine whether that loss is acceptable.

## Ask your AI tutor

```text
Explain this notebook one transformation at a time.
Before each cell ask me to predict shapes, units, and a check.
Give edits in executable cells of at most 20 lines.
Keep the prescribed split, random seed, and tests intact.
Distinguish generated suggestions from executed results.
After the failure experiment, ask me to explain the mechanism.
```

In [1]:
import numpy as np
from scipy.special import expit
from sklearn.linear_model import LogisticRegression
rng=np.random.default_rng(416)
latent=rng.normal(size=(900,3));mix=rng.normal(0,.6,(3,12))
clean=latent@mix;X=clean+rng.normal(0,.15,clean.shape)
y=(latent[:,0]>0).astype(float)
pre,train,test=np.arange(500),np.arange(500,650),np.arange(650,900)
W=rng.normal(0,.1,(12,3));D=rng.normal(0,.1,(3,12));losses=[]
corrupted=X[pre]+rng.normal(0,.2,X[pre].shape)
for step in range(350):
    z=corrupted@W;residual=z@D-clean[pre];losses.append(np.mean(residual**2))
    grad=2*residual/residual.size
    gW=corrupted.T@(grad@D.T);gD=z.T@grad
    W-=.08*gW;D-=.08*gD
print('Denoising reconstruction loss:',losses[0],losses[-1])
assert losses[-1]<losses[0]/3


Denoising reconstruction loss: 1.2068431143441591 0.015291843201967761


In [2]:
frozen_copy=W.copy()
head=LogisticRegression().fit(X[train]@W,y[train])
frozen_score=head.score(X[test]@W,y[test])
assert np.array_equal(W,frozen_copy)
Wfine=W.copy();v=head.coef_.ravel().copy();b=float(head.intercept_[0]);fine_loss=[]
for step in range(150):
    z=X[train]@Wfine;logit=z@v+b;delta=(expit(logit)-y[train])/len(train)
    fine_loss.append(np.mean(np.logaddexp(0,logit)-y[train]*logit))
    gW=X[train].T@(delta[:,None]*v);gv=z.T@delta;gb=delta.sum()
    Wfine-=.03*gW;v-=.03*gv;b-=.03*gb
fine_score=np.mean((expit((X[test]@Wfine)@v+b)>=.5)==y[test])
print('Frozen / fine-tuned test accuracy:',frozen_score,fine_score)
assert fine_loss[-1]<fine_loss[0] and not np.allclose(Wfine,W)
assert frozen_score>.8 and np.array_equal(W,frozen_copy)


Frozen / fine-tuned test accuracy: 0.96 0.96


In [3]:
bad_encoder=np.zeros_like(W)
collapsed=X[test]@bad_encoder
assert np.all(collapsed==0)
print('Collapsed representation unique rows:',np.unique(collapsed,axis=0).shape[0])


Collapsed representation unique rows: 1


## Deliberate failure and repair

A collapsed encoder maps every participant to the same vector, so no head can recover participant-specific information from it. Repair the representation-learning objective and inspect feature variance before interpreting a downstream failure. Do not call the fine-tuning experiment frozen: its copied encoder demonstrably changes.

## Your investigation

List the exact training rows used by the pretraining objective, the frozen head, and fine-tuning. Describe an augmentation that preserves an age-prediction target and one that might damage a lesion-localization target. Compare frozen features with a raw-feature baseline using the same independent evaluation plan.

## Transfer to real neuroimaging

The upstream exercises cover richer neural representations and objectives. Our linear denoising network executes actual pretraining and fine-tuning on synthetic measurements. It is not a pretrained foundation checkpoint, a contrastive-learning reproduction, or an MRI validation study.

**Primary teaching sources, pinned where hosted on GitHub:**

- [NMA DL: unsupervised and self-supervised learning](https://github.com/NeuromatchAcademy/course-content-dl/blob/caba36c513fb8139ac3c9e7503f7a769dadde25e/tutorials/W3D3_UnsupervisedAndSelfSupervisedLearning/student/W3D3_Tutorial1.ipynb)
- [NMA DL: autoencoders](https://github.com/NeuromatchAcademy/course-content-dl/blob/caba36c513fb8139ac3c9e7503f7a769dadde25e/tutorials/W2D3_GenerativeModelsAndDeepLearningDiscussion1/student/W2D3_Tutorial1.ipynb)

Pinned upstream tutorials are a separate assignment; they have **not been executed** by this core lab. They may require data downloads, specialist dependencies, unfinished student cells, and additional compute.

## Exit questions and answer key

1. What is frozen during head training? **Encoder parameters, not the new head or its fitted preprocessing.**
2. Does low reconstruction loss guarantee downstream usefulness? **No; the reconstructed information may not be the target-relevant information.**